In [ ]:
import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
seed = 42

## Model Architecture

In [ ]:
def set_seeds(seed=seed):
    # set random see for tensorflow, keras, numpy etc.
    tf.keras.utils.set_random_seed(seed)
    # only relevant if calculation is done on GPU
    tf.config.experimental.enable_op_determinism()

def create_windows(data, window_size, rollout_length):
    X_list, y_list, Insol_rollout_list = [], [], []
    for i in range(len(data) - window_size - rollout_length):
        X_list.append(data[i:i + window_size,:])
        y_list.append(data[i + window_size: i + window_size + rollout_length, :-1])
        Insol_rollout_list.append(data[i + window_size: i + window_size + rollout_length, -1])

    return np.array(X_list), np.array(y_list), np.array(Insol_rollout_list)


In [ ]:
def CNN_model(window_size, n_features):
    """
    Convolutional Neural Network architecture to be used for the SR-LPNE model
    --------------------------------------------------------------------------

    window_size - input window size
    n_features - number of input variables (here set to 4 for X,Y,Z,Insol

    """
    # layer names are pinned explicitly (matching Keras's own default names for a
    # session's first CNN_model build) so save_weights()/load_weights() - which
    # match variables by layer name - keep working no matter how many times this
    # function has already been called in the current kernel session
    # 0 parameters
    inp = layers.Input(shape=(window_size, n_features))

    x = layers.GaussianNoise(0.05, name="gaussian_noise")(inp) #increased from 0.05

    # 16 filters and 5 kernel size
    x = layers.Conv1D(filters = params['conv filters'],
                      kernel_size = params['conv kernel'], # changed from 5 to 50 for high resolution
                      activation = 'swish',
                      padding = "causal",
                      name = "conv1d")(x)

    x = layers.LayerNormalization(name="layer_normalization")(x)

    # extract last tensor from convolutional layer output
    x = layers.Lambda(lambda x: x[:, -1, :], name="lambda")(x)

    x = layers.Dense(units = params['dense unit'],
                     activation="swish",
                     name = "dense")(x)

    # dropout layer to reduce overfitting; dropout rate is a hyperparameter
    x = layers.Dropout(rate=params['dropout rate'], name="dropout")(x)

    # output layer, returns the vector [\Delta_X,\Delta_Y,\Delta_Z]
    out = layers.Dense(units = 3,
                       activation = 'linear',
                       kernel_initializer = 'zeros',
                       bias_initializer = 'zeros',
                       use_bias=True,
                       name = "dense_1")(x)

    # physical constraint: scaling by maximum incerement in data
    # WARNING: NEEDS tO BE ADJUSTED FOR DIFFERENT DATA SETS
    #out = out * tf.constant([0.47893476,0.78733971,0.21167531], dtype=tf.float32)

    return Model(inputs = inp,outputs = out)


In [ ]:
class RolloutEmulator(tf.keras.Model):
    """
    custom keras model class for rollout training, validation and prediction

    Loss: Harmonized MSE (hMSE) computed on a multi-level (default 2-level) Haar
    DWT of the rollout trajectory, per "The Procrustean Bed of Time Series"
    (arXiv:2512.18610), Eq. 21-23. The DWT is implemented with tf.nn.conv1d using
    fixed (non-trainable) Haar filter taps so it stays differentiable under
    GradientTape - PyWavelets operates outside the TF graph and would break
    gradient flow. Multi-level decomposition is the standard recursive "pyramid
    algorithm": each level decomposes the previous level's approximation into a
    new, shorter approximation plus a detail band; every level's detail band is
    kept, and only the final (coarsest) approximation is kept from the last level.
    """
    def __init__(self,brain,rollout_length,vali_rollout_length,window_size,n_features,
                 L_2_weight=1.0,dwt_gamma=0.5,dwt_beta=0.3,dwt_eps=1e-6,dwt_levels=3):
        """
        initializer method for the rollout trainig class
        ------------------------------------------------

        brain               - model to train via rollout training
        rollout_length      - training rollout length
        vali_rollout_length - validation rollout length
        L_2_weight          - weight on the harmonized MSE loss (default 1.0)
        dwt_gamma           - Eq. 21 gamma: balances the flat vs. relative-error term (paper default 0.5)
        dwt_beta            - Eq. 23 beta: EMA smoothing factor for the historical-magnitude tracker (paper default 0.3)
        dwt_eps             - Eq. 21 epsilon: minimum threshold preventing weight explosion
        dwt_levels           - number of recursive DWT decomposition levels (default 2)
        noise_level         - standard deviation of normally distributed noise used for noise injection during training
        """

        super(RolloutEmulator, self).__init__()
        self.brain = brain
        self.rollout_length = rollout_length
        self.vali_rollout_length = vali_rollout_length
        self.L_2_weight = L_2_weight
        self.noise_level = 1e-2 # change to zero at some point

        self.brain.build((None, window_size, n_features))

        # --- Harmonized MSE (hMSE) setup --------------------------------------
        # Haar wavelet: orthonormal 2-tap low-pass (approximation) / high-pass (detail)
        # filters, shaped (filter_width, in_channels=1, out_channels=1) for conv1d.
        self.dwt_low  = tf.constant([[[1/np.sqrt(2)]], [[ 1/np.sqrt(2)]]], dtype=tf.float32)
        self.dwt_high = tf.constant([[[1/np.sqrt(2)]], [[-1/np.sqrt(2)]]], dtype=tf.float32)

        self.dwt_gamma = dwt_gamma
        self.dwt_beta = dwt_beta
        self.dwt_eps = dwt_eps
        self.dwt_levels = dwt_levels

        # historical-magnitude EMA state (f_bar in Eq. 23), one tracker per DWT
        # coefficient group (each level's detail band, plus the final coarsest
        # approximation), one value per (coefficient index, state variable).
        # train_step and vali_step roll out to different lengths, so each needs
        # its own set of trackers sized to its own coefficient-group lengths.
        # Haar DWT with stride 2 / VALID padding on length L gives L//2
        # coefficients per level (an odd L would floor-truncate the last sample -
        # both rollout lengths here are even at every level this goes to, so this
        # doesn't come up).
        train_group_lengths = self._dwt_multilevel_lengths(self.rollout_length, self.dwt_levels)
        vali_group_lengths = self._dwt_multilevel_lengths(self.vali_rollout_length, self.dwt_levels)

        self.fbar_train = [tf.Variable(tf.ones((L, 3)), trainable=False, name=f"fbar_train_{i}")
                            for i, L in enumerate(train_group_lengths)]
        self.fbar_vali = [tf.Variable(tf.ones((L, 3)), trainable=False, name=f"fbar_vali_{i}")
                           for i, L in enumerate(vali_group_lengths)]

    # returns number of (trainable) variables/ size of the neural network
    @property
    def trainable_variables(self):
        return self.brain.trainable_variables

    # call to model to predict dependent on the inputs (training set to None)
    def call(self, inputs, training=None):
        return self.brain(inputs, training=training)

    def _dwt(self, x, length):
        """
        Shallow (1-level) Haar DWT applied independently to each of the 3 state
        channels (X, Y, Z), via tf.nn.conv1d with the channel dimension folded
        into the batch dimension so channels aren't mixed together.

        x: (batch, length, 3) -> cA, cD: (batch, length//2, 3)
        """
        batch = tf.shape(x)[0]
        coeff_len = length // 2

        x_ch = tf.transpose(x, [0, 2, 1])              # (batch, 3, length)
        x_ch = tf.reshape(x_ch, [-1, length, 1])        # (batch*3, length, 1)

        cA = tf.nn.conv1d(x_ch, self.dwt_low,  stride=2, padding="VALID")  # (batch*3, coeff_len, 1)
        cD = tf.nn.conv1d(x_ch, self.dwt_high, stride=2, padding="VALID")

        cA = tf.transpose(tf.reshape(cA, [batch, 3, coeff_len]), [0, 2, 1])  # (batch, coeff_len, 3)
        cD = tf.transpose(tf.reshape(cD, [batch, 3, coeff_len]), [0, 2, 1])
        return cA, cD

    def _dwt_multilevel(self, x, length, levels):
        """
        Recursive multi-level (Mallat pyramid) Haar DWT, built on top of the
        single-level _dwt above: at each level, the current approximation is
        decomposed into a new, shorter approximation plus a detail band; the
        detail band is kept and the approximation is fed into the next level.

        x: (batch, length, 3)
        returns: a list of (batch, len_i, 3) tensors, ordered
                 [cD_level1, cD_level2, ..., cD_levelN, cA_levelN]
                 i.e. detail coefficients from finest to coarsest scale, followed
                 by the final (coarsest) approximation.
        """
        coeffs = []
        current, current_length = x, length
        for _ in range(levels):
            cA, cD = self._dwt(current, current_length)
            coeffs.append(cD)
            current, current_length = cA, current_length // 2
        coeffs.append(current)  # final coarse approximation
        return coeffs

    @staticmethod
    def _dwt_multilevel_lengths(length, levels):
        """Coefficient-group lengths for _dwt_multilevel, in the same order it returns them."""
        lengths = []
        current_length = length
        for _ in range(levels):
            current_length = current_length // 2
            lengths.append(current_length)
        lengths.append(current_length)  # final approximation shares the last level's length
        return lengths

    def _harmonized_mse(self, true_seq, pred_seq, length, fbar_list, update):
        """
        Harmonized MSE (Eq. 21) computed on a multi-level Haar DWT of true_seq/pred_seq.

        true_seq, pred_seq - (batch, length, 3) ground-truth / predicted trajectories
        fbar_list           - list of (len_i, 3) EMA trackers, one per DWT coefficient
                               group (see _dwt_multilevel for the group ordering)
        update               - whether to update fbar_list with this batch (Eq. 23)
        """
        true_groups = self._dwt_multilevel(true_seq, length, self.dwt_levels)
        pred_groups = self._dwt_multilevel(pred_seq, length, self.dwt_levels)

        loss = 0.0
        for true_g, pred_g, fbar in zip(true_groups, pred_groups, fbar_list):
            if update:
                batch_mag = tf.reduce_mean(tf.abs(true_g), axis=0)  # (len_i, 3)
                fbar.assign(self.dwt_beta * fbar + (1 - self.dwt_beta) * batch_mag)

            weight = 1.0 + self.dwt_gamma / (fbar + self.dwt_eps)  # (len_i, 3)
            loss += tf.reduce_mean(weight * tf.square(true_g - pred_g))

        return loss

    # heart of the actual training step
    @tf.function
    def train_step(self,data):

        # as you can see here, the training step will recieve data in a tuple of (features,data), which is equivalent to the data you pass to the
        # x= ... and y= .. argument in the keras.Model.fit(...) method
        (features,y) = data

        # features itself contains the X_init values being the values of the state space (X,Y,Z) and the insolation forcing in the input window w at index 0
        # at index 1 the "future" insolation forcing aka. the insolation during the rollout is passed with shape (rollout_length,1)
        X_init = features[0]
        future_forcing = features[1]

        batch_size = tf.shape(X_init)[0]
        n_features = tf.shape(X_init)[2]

        # scale augmentation to randomly rescale state space feature and target trajectories
        scale = tf.random.uniform([batch_size,1,1], 0.8, 1.2)

        # apply scaling to (X,Y,Z) in the input window
        X_init_scaled = tf.concat([X_init[:, :, 0:3] * scale, X_init[:, :, 3:]], axis=-1)
        y_scaled = y * scale

        # tell tensorflow to create the context manager of the variables created in the with statement and tracks them to calculate the gradients
        with tf.GradientTape() as tape:

            # passing input window as first window to use in rollout training
            current_window = X_init_scaled

            # calculate ground truth trajectory
            true_path = tf.concat([X_init_scaled[:, -1:, 0:3], y_scaled], axis=1)

            all_paths = []

            # actual rollout training
            for i in range(self.rollout_length):

                # current window (first step this is just the input window) with added normally distributed (input) noise
                noisy_window = current_window + tf.random.normal(tf.shape(current_window), stddev=self.noise_level)

                # generating increment predictions (forward-pass)
                d_prediction = self.brain(noisy_window, training=True)

                # adding predicted increment to last entry in the crurrent windoe
                x_now = current_window[:, -1, 0:3]
                prediction_abs = x_now + d_prediction

                all_paths.append(prediction_abs)

                # adding additional output noise
                prediction_abs += tf.random.normal(tf.shape(prediction_abs), stddev=self.noise_level)

                next_step = tf.zeros((batch_size, 1, n_features))

                pred = tf.expand_dims(prediction_abs, axis=1)

                forc_true = future_forcing[:, i]

                # noise injection usedto prevent NN from remembering just the values, resulting in insolation just being used as a clock (safety measure)
                forc_noisy = forc_true + tf.random.normal(tf.shape(forc_true), stddev=self.noise_level)

                forc = tf.reshape(forc_noisy, (batch_size, 1, 1))
                next_step = tf.concat([pred,forc],axis=-1)

                # appending next predicted state and insolation forcig to the current window and moving window one time step forward
                current_window = tf.concat([current_window[:,1:,:],next_step],axis=1)

            # creating stack over rollout window for trajectory
            p_stack = tf.stack(all_paths, axis=1)

            # Harmonized MSE, computed on a shallow Haar DWT of the trajectory (Eq. 21)
            abs_loss = self._harmonized_mse(true_path[:,1:,:], p_stack, self.rollout_length,
                                             self.fbar_train, update=True)
            total_loss = self.L_2_weight * abs_loss

            # plain point-to-point MSE, logged for comparison only - not part of total_loss,
            # so it has no effect on the gradients computed below
            pointwise_mse = tf.reduce_mean(tf.square(true_path[:,1:,:] - p_stack))

        # Backporpagation through the forward pass operations tracked by GradientTape()
        grads = tape.gradient(total_loss,self.trainable_variables)
        # weight update
        self.optimizer.apply_gradients(zip(grads,self.trainable_variables))

        return {
            "loss": total_loss,
            "abs": abs_loss,
            "mse": pointwise_mse}

    # test_step used in the validation process
    # essentially the same as the training step, only with training disabled and no backpropagation/ weights update
    def test_step(self,data):

        def calculate_lag1_acf(p_stack):
            """
            calculate the lag_1 auto-correlation function to ensure causality in the validation step is preserved
            --------------------------------------------------

            p_stack - predicted validation rollout trajetory
            """

            mu = tf.reduce_mean(p_stack, axis=1, keepdims=True)
            var = tf.math.reduce_variance(p_stack, axis=1, keepdims=True)

            p_centered = p_stack - mu
            t0 = p_centered[:, :-1, :]
            t1 = p_centered[:, 1:, :]

            acf_per_sample = tf.reduce_mean(t0 * t1, axis=1, keepdims=True) / (var + 1e-7)

            return tf.reduce_mean(acf_per_sample)


        (features,y) = data
        X_init = features[0]
        future_forcing = features[1]

        batch_size = tf.shape(X_init)[0]
        n_features = tf.shape(X_init)[2]

        current_window = X_init

        all_paths = []

        true_path = tf.concat([X_init[:, -1:, 0:3], y], axis=1) # [Batch, Rollout+1, 3]

        for i in range(self.vali_rollout_length):
            d_prediction = self.brain(current_window, training=False)

            x_now = current_window[:, -1, 0:3]
            prediction_abs = x_now + d_prediction

            all_paths.append(prediction_abs)

            next_step = tf.zeros((batch_size, 1, n_features))

            pred = tf.expand_dims(prediction_abs, axis=1)
            forc = tf.reshape(future_forcing[:, i], (batch_size, 1, 1))
            next_step = tf.concat([pred,forc],axis=-1)

            current_window = tf.concat([current_window[:,1:,:],next_step],axis=1)

        p_stack = tf.stack(all_paths, axis=1)

        # Harmonized MSE on the validation-length rollout - its own EMA tracker,
        # since the coefficient count differs from the training rollout length.
        # Updated here too (each validation pass) so its weighting is actually
        # adaptive rather than frozen at the initial f_bar=1 value; the paper
        # describes the EMA as updating "at each training iteration," so switch
        # update=False here if you want strictly training-only magnitude tracking.
        abs_loss = self._harmonized_mse(true_path[:,1:,:], p_stack, self.vali_rollout_length,
                                         self.fbar_vali, update=True)

        # plain point-to-point MSE, logged for comparison only
        pointwise_mse = tf.reduce_mean(tf.square(true_path[:,1:,:] - p_stack))

        lag1_acf = calculate_lag1_acf(p_stack)

        # calculate standard deviation for ground truth and prediction in validation step
        y_std = tf.math.reduce_std(y,axis = 1)
        p_std = tf.math.reduce_std(p_stack, axis=1)

        # calculate C_2 constraint
        C_2 =  tf.reduce_mean(tf.abs(1.0 - (p_std / (y_std + 1e-6))))

        total_loss = self.L_2_weight * abs_loss

        return {
            "loss": total_loss,
            "abs": abs_loss,
            "mse": pointwise_mse,
            "lag1_acf": lag1_acf,
            "C_2": C_2}

In [ ]:
class PhysicalAttractorCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self,filepath="best_state_space_model.weights.h5",acf_threshold=0.97):
        """
        initializer method for C_2 constraint callback
        ------------------------------------------------

        filepath               - file path to save the best model weights
        acf_threshold          - threshold fro acf_function to accept model
        """
        super(PhysicalAttractorCheckpoint,self).__init__()
        self.filepath = filepath
        self.acf_threshold = acf_threshold
        self.best_C_2 = float('inf')
        self.best_epoch = 0
        self.patience = 0

    def on_epoch_end(self,epoch=0,logs=None):

        # retrieve current C_2 and lag_1 ACF values
        current_C_2 = logs.get("val_C_2")
        current_acf = logs.get("val_lag1_acf")

        if current_C_2 is None or current_acf is None:
            print("\nWarning: Physical metrics not found in logs. Check test_step return keys.")
            return

        # convert from string to float
        current_C_2 = float(current_C_2)
        current_acf = float(current_acf)

        # check if ACF criteria is meet
        if current_acf < self.acf_threshold:
            if (epoch + 1) % 5 == 0:
                print(f"\nEpoch {epoch+1}: Smoothness ({current_acf:.4f}) below threshold. Ignoring.")
            self.patience += 1
            if self.patience == 15:
                self.model.stop_training = True
                print(f"\nPatience ran out. (Should) Early stopping at epoch {epoch}: no physically-valid "
                      f"improvement in val_C_2 for {self.patience} epochs.")
            return

        # check if current model returns a lower C_2 constraint then previous model, if yes overwrite the current best model
        if current_C_2 < self.best_C_2:
            self.best_C_2 = current_C_2
            self.best_epoch = epoch
            self.model.save_weights(self.filepath)
            self.patience = 0
            print(f"\nEpoch {epoch+1}: NEW PHYSICAL BEST!")
            print(f" > C_2: {current_C_2:.4f}")
            print(f" > Lag-1 ACF:    {current_acf:.4f}")
            print(f" > Weights saved to {self.filepath}")
        else:
            self.patience += 1
            if self.patience == 15:
                self.model.stop_training = True
                print(f"\nPatience ran out. (Should) Early stopping at epoch {epoch}: no physically-valid "
                      f"improvement in val_C_2 for {self.patience} epochs.")


class WeightStatsLogger(tf.keras.callbacks.Callback):
    """Prints min/max/std of the harmonized-MSE weight tensors (1 + gamma/(fbar+eps))
    at the end of every epoch, for both the training- and validation-length trackers.
    Pools the weights across every DWT coefficient group (all decomposition levels)."""

    def _weight_stats(self, fbar_list):
        gamma, eps = self.model.dwt_gamma, self.model.dwt_eps
        weights = tf.concat([tf.reshape(1.0 + gamma / (fbar + eps), [-1]) for fbar in fbar_list], axis=0)
        return tf.reduce_min(weights), tf.reduce_max(weights), tf.math.reduce_std(weights)

    def on_epoch_end(self, epoch, logs=None):
        train_min, train_max, train_std = self._weight_stats(self.model.fbar_train)
        vali_min, vali_max, vali_std = self._weight_stats(self.model.fbar_vali)
        print(f" > hMSE weights (train): min={train_min:.4f}, max={train_max:.4f}, std={train_std:.4f}")
        print(f" > hMSE weights (vali):  min={vali_min:.4f}, max={vali_max:.4f}, std={vali_std:.4f}")


## Data Loading and Config

In [ ]:
# SM90 reference run - plotted in grey for visual reference only.
# NOTE on the SM90 csvs (see Saltzman_Model.ipynb): in the X/Y/Z files the `age_kyrBP`
# column is actually MODEL TIME since the start of the integration (forcing is R(t_cutoff - t)),
# so the true age is 4000 - age_kyrBP; the Insol file's `age_kyrBP` IS the true age.
importpath = "/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/SM90/4000kyr_SM90_"
_sm90_X = pd.read_csv(f"{importpath}X.csv")
sm90_age = 4000.0 - _sm90_X['age_kyrBP'].to_numpy(dtype=float)
sm90_state = np.column_stack([
    _sm90_X['X'].to_numpy(dtype=float),
    pd.read_csv(f"{importpath}Y.csv")['Y'].to_numpy(dtype=float),
    pd.read_csv(f"{importpath}Z.csv")['Z'].to_numpy(dtype=float),
])
_order = np.argsort(sm90_age)          # np.interp needs increasing x
sm90_age, sm90_state = sm90_age[_order], sm90_state[_order]

def sm90_at(ages):
    """SM90 X, Y, Z interpolated onto `ages` (ka BP) and z-scored over that span, since SM90 is in
    its own non-dimensional units and can't go through the observational scaler."""
    out = np.column_stack([np.interp(ages, sm90_age, sm90_state[:, k]) for k in range(3)])
    return (out - out.mean(axis=0)) / out.std(axis=0)


In [ ]:
# ---- which trained run to evaluate ------------------------------------------
import sys, importlib
sys.path.insert(0, '/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Scripts')
import run_config as rc
importlib.reload(rc)

DATASET = '1243ka'
RUN_TAG = None          # None -> newest trained run for DATASET (see rc.list_runs(DATASET)), or paste a run tag
LEGACY_WEIGHTS = None   # path to an old weights file that has no saved config (e.g. '.../ka1461_model.weights.h5').
                        # Its scaler is then REFIT with the shared split convention - add test_size=... to
                        # make_config below if that old run used a different split. Provenance of old files
                        # can't be checked, so prefer retraining with DWT-LPNE.

if LEGACY_WEIGHTS is None:
    RUN_TAG = RUN_TAG or rc.latest_run(DATASET)
    cfg, train_scaler, paths = rc.load_run(RUN_TAG)   # the exact scaler the model was trained with
    weights_path = paths['weights']
else:
    cfg = rc.make_config(DATASET)
    train_scaler = None
    weights_path = LEGACY_WEIGHTS

params = cfg['params']

# observational data, ordered oldest -> youngest (row 0 = oldest), same split as in training
dat = rc.load_data(cfg)
splits = rc.split_data(dat, cfg)
if train_scaler is None:
    train_scaler = rc.fit_scaler(splits)

print("run    :", cfg['run_tag'])
print("weights:", weights_path)
print(rc.describe_split(splits))

time = splits['time']
test_size = splits['test_size']
data = train_scaler.transform(splits['full'])

# scale the GP std the same way as data - only divide by the fitted per-feature scale_,
# no mean subtraction, since a spread isn't shifted by StandardScaler's centering step
data_std_scaled = splits['full_std'] / train_scaler.scale_[:3]

fig, axes = plt.subplots(3, 1, figsize=(20,6),sharex=True)
for i in range(len(axes)):
    axes[i].plot(time, data[:,i])
    axes[i].fill_between(time, data[:,i] - 2*data_std_scaled[:,i], data[:,i] + 2*data_std_scaled[:,i], alpha=0.4)
axes[-1].invert_xaxis()
axes[-1].set_xlabel("Age [ka BP]")


In [ ]:
# set random initilization seed
set_seeds(cfg['seed'])
plt.rcParams.update({'font.size': 20})

# window / rollout lengths come from the run's config (defined in kyr in run_config.make_config)
timestep = cfg['timestep']  # kyr
window_size = cfg['window_size']
rollout_length = cfg['rollout_length']
rollout_length_validation = cfg['rollout_length_validation']

mappings = ["X","Y","Z"]

color_ls = [["#D6195E","#FFB600"],["#1E88E5","#6519C5"],["#05EFC5","#8DB1A7"]]

# hyperparameters (`params`) are loaded with the run config in the cell above


## Model Rollout

In [ ]:
brain = CNN_model(window_size=window_size, n_features=4)

model = RolloutEmulator(
    brain,
    rollout_length=rollout_length,
    vali_rollout_length=rollout_length_validation,
    window_size=window_size,
    n_features=4,
    dwt_gamma=params['dwt gamma'],
    dwt_beta=params['dwt beta'],
    dwt_eps=params['dwt eps'],
    dwt_levels=params['dwt levels'],
)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=params['learning rate'], clipnorm=1.0))

# subclassed model - variables don't exist until it's called once, so build with a dummy
# input of the right shape before load_weights can restore anything into it
dummy_X = tf.zeros((1, window_size, 4))
_ = model(dummy_X)

print("Loading best physical model for final evaluation...")
model.load_weights(str(weights_path))


In [ ]:
# data is ordered oldest -> youngest (run_config.load_data), same as in training, so the
# seed window is simply the oldest window_size rows
IC = data[:window_size]
IC_CIs = data_std_scaled[:window_size]
randomizer = npr.default_rng(seed = cfg['seed']+3)

def createrandomstate(IC, CI, rng):
    ICnew = np.zeros((IC.shape[0], IC.shape[1]))
    for i in range(IC.shape[0]):
        ICnew[i,0] = IC[i,0] + rng.normal(0, CI[i,0])
        ICnew[i,1] = IC[i,1] + rng.normal(0, CI[i,1])
        ICnew[i,2] = IC[i,2] + rng.normal(0, CI[i,2])
        ICnew[i,3] = IC[i,3]  # insolation forcing has no GP uncertainty - keep it exact

    return ICnew


## Monte Carlo Rollout


In [ ]:
@tf.function
def predict_step_batch(w):
    return model(w, training=False)

# data is already oldest -> youngest, matching the seed window
full_data_seed = data
full_time_seed = time
full_std_seed = data_std_scaled

n_mc = 1000
total_steps = full_data_seed.shape[0] - window_size

# 1000 independently perturbed seed windows, batched: (n_mc, window_size, 4)
seed_windows = np.stack([createrandomstate(IC, IC_CIs, randomizer) for _ in range(n_mc)])
current_window = tf.convert_to_tensor(seed_windows, dtype=tf.float32)

# autoregressive rollout, same mechanics as the DWT-LPNE test-set rollout (predict
# the increment, add it to the last state, splice in the true forcing for the next
# step) but batched over all 1000 ensemble members at once instead of looping per
# member - the forcing is deterministic ground truth, shared across the ensemble
mc_pred = np.zeros((n_mc, total_steps, 4), dtype=np.float32)

for t in tqdm(range(total_steps), desc="MC ensemble rollout"):
    nxt_increments = predict_step_batch(current_window)
    nxt_state = current_window[:, -1, 0:3] + nxt_increments

    forcing = tf.fill([n_mc, 1], full_data_seed[window_size + t, -1])
    next_step = tf.concat([nxt_state, tf.cast(forcing, tf.float32)], axis=-1)

    mc_pred[:, t, :] = next_step.numpy()
    current_window = tf.concat([current_window[:, 1:, :], next_step[:, tf.newaxis, :]], axis=1)

# prepend each ensemble member's own (perturbed) seed window - the fixed initial
# condition the emulator was actually fed before autoregressively rolling out -
# so the plotted trajectories show it instead of starting right after it
mc_rollout = np.concatenate([seed_windows, mc_pred], axis=1)
mc_rollout_time = full_time_seed

# marks where the fixed IC window ends and the autoregressive rollout begins
rollout_start_time = full_time_seed[window_size]

# short/test-set span now includes the seed window itself, since the seed is the
# oldest window_size rows of the record, which lie within the oldest test_size rows
test_steps = test_size


In [ ]:
SHORT_PCT = (5, 95)      # ensemble band, short (test-set) panels
LONG_PCT = (5, 95)       # ensemble band, long panels (was 0.1-99.9 while labelled 5-95%)

# the ensemble can be multimodal (separate trajectory bundles), so the mean can be a
# curve no member follows - plot the median as the central line instead
sm90_ref = sm90_at(mc_rollout_time)   # z-scored SM90, aligned by true age

fig, axs = plt.subplots(3, 2, figsize=(20, 12), sharex='col')

for i in range(3):
    # left column: short rollout, test set only (includes the IC window)
    ax = axs[i, 0]
    test_time_plot = mc_rollout_time[:test_steps]
    ens = mc_rollout[:, :test_steps, i]
    median = np.median(ens, axis=0)
    lower, upper = np.percentile(ens, SHORT_PCT, axis=0)
    ax.plot(test_time_plot, median, color=color_ls[i][1], label="MC median")
    ax.fill_between(test_time_plot, lower, upper, color=color_ls[i][1], alpha=0.25, label=f"{SHORT_PCT[0]}-{SHORT_PCT[1]}%")
    ax.plot(test_time_plot, full_data_seed[:test_steps, i],
            color=color_ls[i][0], linestyle="dashed", alpha=0.6, label="ground truth")
    ax.plot(test_time_plot, sm90_ref[:test_steps, i],
            alpha = 0.25, color = 'black', label="SM90 (z-scored)")
    ax.axvline(rollout_start_time, color="gray", linestyle=":", alpha=0.6, label="IC / rollout boundary")
    ax.invert_xaxis()
    ax.set_ylabel(mappings[i])
    ax.legend(fontsize=8, loc="upper right")
    if i == 0:
        ax.set_title("Short rollout (test set)")
    if i == 2:
        ax.set_xlabel("Age [ka BP]")

    # right column: long rollout, whole ground truth (includes the IC window)
    ax = axs[i, 1]
    ens = mc_rollout[:, :, i]
    median = np.median(ens, axis=0)
    lower, upper = np.percentile(ens, LONG_PCT, axis=0)
    ax.plot(mc_rollout_time, median, color=color_ls[i][1], label="MC median")
    ax.fill_between(mc_rollout_time, lower, upper, color=color_ls[i][1], alpha=0.25, label=f"{LONG_PCT[0]}-{LONG_PCT[1]}%")
    ax.plot(mc_rollout_time, full_data_seed[:, i],
            color=color_ls[i][0], linestyle="dashed", alpha=0.6, label="ground truth")
    ax.plot(mc_rollout_time, sm90_ref[:, i],
            alpha = 0.25, color = 'black', label="SM90 (z-scored)")
    ax.axvline(rollout_start_time, color="gray", linestyle=":", alpha=0.6, label="IC / rollout boundary")
    ax.invert_xaxis()
    ax.legend(fontsize=8, loc="upper right")
    if i == 0:
        ax.set_title("Long rollout (whole ground truth)")
    if i == 2:
        ax.set_xlabel("Age [ka BP]")

plt.tight_layout()
plt.show()


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(20, 12), sharex=True)

mc_median = np.median(mc_rollout, axis=0)
test_time_plot = mc_rollout_time[:test_steps]

for i in range(3):
    ax = axs[i]
    for m in range(mc_rollout.shape[0]):
        ax.plot(test_time_plot, mc_rollout[m, :test_steps, i], color='black', alpha=0.01)
    ax.plot(test_time_plot, full_data_seed[:test_steps, i],
            color=color_ls[i][0], linestyle="dashed", alpha=0.8, label="ground truth")
    ax.fill_between(test_time_plot,
                     full_data_seed[:test_steps, i] - 2 * full_std_seed[:test_steps, i],
                     full_data_seed[:test_steps, i] + 2 * full_std_seed[:test_steps, i],
                     color=color_ls[i][0], alpha=0.2, label="ground truth 95% CI")
    ax.plot(test_time_plot, mc_median[:test_steps, i], color=color_ls[i][1], label="MC median")
    ax.axvline(rollout_start_time, color="gray", linestyle=":", alpha=0.6, label="IC / rollout boundary")
    ax.invert_xaxis()
    ax.set_ylabel(mappings[i])
    ax.legend(fontsize=8, loc="upper right")

axs[0].set_title("Individual MC trajectories vs. ground truth (test set)")
axs[-1].set_xlabel("Age [ka BP]")
plt.tight_layout()
plt.show()


In [ ]:
ic_time = time[:window_size]

fig, axes = plt.subplots(3, 1, figsize=(20,6),sharex=True)
for i in range(len(axes)):
    axes[i].plot(ic_time, IC[:,i], color = 'red')
    axes[i].fill_between(ic_time, IC[:,i] - 2*IC_CIs[:,i], IC[:,i] + 2*IC_CIs[:,i], 
                         color = 'red', alpha=0.4)
    for j in range(1000):
        axes[i].plot(ic_time, createrandomstate(IC, IC_CIs, randomizer)[:,i], 
                color = 'black', alpha = 0.005)
axes[-1].invert_xaxis()
